In [68]:
!pip install -q -U keras-hub
!pip install  -q -U keras
!pip install -q -U keras-nlp

In [72]:
import pandas as pd
import pyarrow as pa
import keras
import keras_hub
import keras
import keras_nlp
from keras_nlp.samplers import TopKSampler
from time import time
import csv

In [73]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/" + splits["train"])

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6678d555-2dc7-4757-bcc3-6022d57f64c4)')' thrown while requesting GET https://huggingface.co/datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


In [74]:
def split_text(row):
    prompt = row.split("[/INST]")[0].replace("<s>[INST]", "").strip()
    response = row.split("[/INST]")[1].replace("</s>", "").strip()
    return prompt, response


df[["prompt", "response"]] = df["text"].apply(lambda x: pd.Series(split_text(x)))

prompts = df["prompt"].tolist()
responses = df["response"].tolist()

data = {
    "prompts": prompts,
    "responses": responses
}

In [82]:
import os 

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"
# avoid memory fragmentation on JAX backend.
os.environ["JAX_PLATFORMS"] = ""

In [83]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("/kaggle/input/gemma3/keras/gemma3_instruct_270m/4")
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 640)         │     268,098,176 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     167,772,160 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 268,098,176 (1022.71 MB)

 Trainable params: 268,098,176 (1022.71 MB)

 Non-trainable params: 0 (0.00 B)

In [85]:
# template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

# prompt = template.format(
#     instruction="What should I do on a trip to Europe?",
#     response="",
# )
# sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
# gemma_lm.compile(sampler=sampler)
# print(gemma_lm.generate(prompt, max_length=256))

TypeError: in user code:

    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/models/gemma3/gemma3_causal_lm.py", line 301, in generate_step  *
        token_ids = self.sampler(
    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/samplers/sampler.py", line 122, in __call__  *
        prompt, _, _, _ = self.run_loop(

    TypeError: outer_factory.<locals>.inner_factory.<locals>.tf__compute_probabilities() got an unexpected keyword argument 'loop_vars'


In [89]:
# Enable LoRA for the model and set the LoRA rank to 5.
gemma_lm.backbone.enable_lora(rank=5)

ValueError: lora is already enabled. This can only be done once per layer.

In [90]:
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 640)         │     268,432,256 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     167,772,160 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 268,432,256 (1023.99 MB)

 Trainable params: 168,320 (657.50 KB)

 Non-trainable params: 268,263,936 (1023.35 MB)

In [ ]:
gemma_lm.fit(data, epochs=1, batch_size=1)

     5/209606 ━━━━━━━━━━━━━━━━━━━━ 640:19:48 11s/step - loss: 1.3958 - sparse_categorical_accuracy: 0.2847